# 🧠 AI Leadership Insight & Decision Agent — Interactive Tester

This notebook allows you to test the core backend logic (RAG pipeline and LangGraph agent) directly in Python, without needing to spin up the Streamlit UI.

**Prerequisites:** You must have already run `python scripts/ingest.py` (or `make ingest`) to populate the ChromaDB and BM25 indexes with your documents.

In [2]:
import logging
import sys
from pathlib import Path

# Set up logging so we can see the Agent's internal thought process
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

# Add the src directory to Python path so we can import our package
src_path = str(Path.cwd() / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from insight_agent.config import Config
from insight_agent.retrieval.retriever import retrieve
from insight_agent.reasoning.generator import generate_answer
from insight_agent.reasoning.agent import run_agent

# Initialize config (this will load from config.yaml and .env automatically)
cfg = Config()
print(f"\n✅ Environment loaded successfully!")
print(f"- Primary LLM: {cfg.llm_model}")
print(f"- Fast LLM (Routing/Critique): {cfg.fast_llm_model}")
print(f"- Embedding Model: {cfg.embed_model}")


✅ Environment loaded successfully!
- Primary LLM: gemini-2.5-flash
- Fast LLM (Routing/Critique): gemini-2.5-flash
- Embedding Model: models/gemini-embedding-001


--- 
## ⚙️ Task 1: Insight Agent (Standard Hybrid RAG)
Ask a direct analytical question. The agent will retrieve relevant chunks via Reciprocal Rank Fusion (Chroma + BM25) and generate a structured analytical report.

In [3]:
task1_question = "What is our current revenue trend and how does APAC factor into it?"

print(f"🔎 Running Insight Agent for: '{task1_question}'\n")
print("-" * 80)

chunks = retrieve(task1_question)
report = generate_answer(task1_question, chunks)

print("-" * 80)
print(report)

🔎 Running Insight Agent for: 'What is our current revenue trend and how does APAC factor into it?'

--------------------------------------------------------------------------------


INFO: Retrieval: dense=10, sparse=10 → fused top 5
INFO: Calling LLM (gemini-2.5-flash) for question: What is our current revenue trend and how does APAC factor i...


--------------------------------------------------------------------------------
## SUMMARY
Our company experienced strong revenue growth throughout FY2024, with total revenue reaching $2.4B, but saw a deceleration in Q4 with flat QoQ growth. The APAC segment significantly factored into this trend, underperforming its FY2024 target by $50M and contributing to the Q4 slowdown due to deal delays and competitive pressures.

## KEY FINDINGS
*   Total FY2024 revenue was $2.4B, with quarterly revenue progressing from $530M in Q1 to $640M in Q3 and Q4.
*   Q4 FY2024 revenue growth decelerated


--- 
## 🤖 Task 2: Decision Agent (LangGraph Reasoning)
Ask a complex strategic question. The LangGraph agent will:
1. **Decompose** the question into sub-questions.
2. **Retrieve** documents for each sub-question.
3. **Synthesize** a draft strategy.
4. **Reflect** on its own draft for weaknesses.
5. Automatically **Loop back to retrieval** if it needs more data.
6. Return the **Final Recommendation**.

In [4]:
task2_question = "Should we expand into Southeast Asia in FY2025 given the current APAC performance?"

print(f"🤖 Running Decision Agent for: '{task2_question}'\n")
print("-" * 80)

answer = run_agent(task2_question)

print("\n" + "=" * 80)
print("FINAL STRATEGIC RECOMMENDATION")
print("=" * 80 + "\n")
print(answer)

INFO: [agent] Starting Decision Agent for: Should we expand into Southeast Asia in FY2025 given the current APAC ...


🤖 Running Decision Agent for: 'Should we expand into Southeast Asia in FY2025 given the current APAC performance?'

--------------------------------------------------------------------------------


INFO: [agent] Decomposed into 3 sub-questions: ['What is the market potential, competitive landscape, and strategic fit of Southeast Asia for our business in FY2025?', 'What are the key drivers and implications of our current APAC performance, and how do they affect our organizational capacity and strategic priorities for a new expansion?', 'What are the projected financial returns, risks, and resource requirements of a Southeast Asia expansion, and how do they compare to alternative investment opportunities?']
INFO: [agent] Retrieved 5 new chunks for: What is the market potential, competitive landscap...
INFO: [agent] Retrieved 5 new chunks for: What are the key drivers and implications of our c...
INFO: [agent] Retrieved 5 new chunks for: What are the projected financial returns, risks, a...
INFO: [agent] Total new unique chunks this pass: 7
INFO: [agent] Draft synthesised (4461 chars)
INFO: [agent] Reflect verdict (iter 0): NO: What is the projected ROI or profitability of this hybr


FINAL STRATEGIC RECOMMENDATION

('## RECOMMENDATION\nYes, Acme Corporation should proceed with expansion into Southeast Asia (SEA) in FY2025. We recommend adopting the Strategy team\'s proposed hybrid approach: initiating organic expansion in Singapore in Q2 FY2025 to establish brand presence, while simultaneously pursuing a strategic acquisition of a regional SaaS player to accelerate market penetration and revenue contribution. This dual strategy is critical to capitalize on SEA\'s significant growth potential and address current APAC underperformance.\n\n## EVIDENCE\n*   **Strategic Imperative:** The APAC segment underperformed its FY2024 target by $50M ($340M vs $390M) and saw market share decline from 9.1% to 8.2% (Source 3, 4). Despite this, SEA is identified as the "fastest growing subregion," "underpenetrated, high upside," and represents Acme\'s "highest long-term growth potential" within APAC (Source 1, 4).\n*   **Market Opportunity:** Southeast Asia has an estimated Total A